# LSTM v2 — Predicción y recomendación de horarios de publicación

Este cuaderno entrena un **modelo general para Facebook** usando el histórico de `dataset_facebook.csv`. La LSTM procesa las últimas publicaciones como una secuencia y combina ese historial con el día y la hora de una publicación candidata.

Al finalizar se genera un ZIP con el modelo, los escaladores, el contrato de características, las métricas y un ranking de demostración. Ese paquete será consumido posteriormente por una API de inferencia; Laravel no tendrá que entrenar el modelo.

> Alcance honesto: el dataset actual no contiene `account_id`, `platform` ni alcance por publicación. Por ello esta versión aprende un patrón general de Facebook y predice un score absoluto de engagement. La personalización por cuenta se añadirá durante la integración y mejorará cuando se acumulen datos reales comparables.

## Diseño metodológico

- **Secuencia histórica:** últimas 7 publicaciones.
- **Datos históricos:** reacciones, comentarios, clics, usuarios que interactuaron, engagement, variables temporales y separación entre publicaciones.
- **Candidato:** día, hora y distancia temporal respecto de la última publicación.
- **Objetivo:** engagement real de la publicación siguiente.
- **Separación:** 70 % entrenamiento, 15 % validación y 15 % prueba, siempre en orden cronológico.
- **Control de fuga:** los escaladores y promedios base se ajustan exclusivamente con entrenamiento.
- **Comparación:** media global y promedio histórico por día/hora contra LSTM.

La fórmula del dataset es: `engagement_score = reacciones + 2 × comentarios + clicks`.

In [ ]:
# Colab ya incluye TensorFlow, pandas, NumPy y scikit-learn.
!pip install -q joblib seaborn
print("Dependencias listas.")

In [ ]:
import json
import os
import random
import shutil
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import tensorflow as tf
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

SEED = 623
WINDOW = 7
EPOCHS = 150
BATCH_SIZE = 32
MODEL_VERSION = "facebook_lstm_v2.0.0"
ARTIFACT_DIR = Path("/content/lstm_horarios_v2_artifacts")

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))

## 1. Cargar el CSV

Ejecuta la siguiente celda y selecciona `dataset_facebook.csv`. No se utiliza Google Drive ni se guardan credenciales dentro del cuaderno.

In [ ]:
from google.colab import files

EXPECTED_FILENAME = "dataset_facebook.csv"
dataset_path = Path("/content") / EXPECTED_FILENAME

if not dataset_path.exists():
    print(f"Selecciona {EXPECTED_FILENAME}")
    uploaded = files.upload()
    if EXPECTED_FILENAME not in uploaded:
        if len(uploaded) != 1:
            raise ValueError(f"Debes subir un único archivo llamado {EXPECTED_FILENAME}.")
        uploaded_name = next(iter(uploaded))
        Path("/content", uploaded_name).replace(dataset_path)

print("Dataset:", dataset_path)
print("Tamaño (bytes):", dataset_path.stat().st_size)

## 2. Validación y limpieza reproducible

Esta sección verifica columnas, tipos, fechas, día de semana y fórmula del objetivo. Los registros se ordenan cronológicamente y los duplicados exactos se eliminan.

In [ ]:
REQUIRED_COLUMNS = [
    "fecha_publicacion", "hora_publicacion", "dia_semana",
    "reacciones", "comentarios", "clicks",
    "usuarios_interactuaron", "engagement_score",
]
NUMERIC_COLUMNS = REQUIRED_COLUMNS[1:]

df = pd.read_csv(dataset_path)
missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

df = df[REQUIRED_COLUMNS].copy()
df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"], errors="coerce")
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

invalid_rows = df[REQUIRED_COLUMNS].isna().any(axis=1).sum()
if invalid_rows:
    print(f"Advertencia: se eliminarán {invalid_rows} filas inválidas.")
df = df.dropna(subset=REQUIRED_COLUMNS).drop_duplicates().copy()

if not df["hora_publicacion"].between(0, 23).all():
    raise ValueError("hora_publicacion debe estar entre 0 y 23.")
if not df["dia_semana"].between(0, 6).all():
    raise ValueError("dia_semana debe estar entre 0 (lunes) y 6 (domingo).")
if (df[["reacciones", "comentarios", "clicks", "usuarios_interactuaron", "engagement_score"]] < 0).any().any():
    raise ValueError("Las métricas de interacción no pueden ser negativas.")

df["timestamp"] = df["fecha_publicacion"].dt.normalize() + pd.to_timedelta(df["hora_publicacion"], unit="h")
calendar_day = df["timestamp"].dt.dayofweek
day_mismatches = int((df["dia_semana"].astype(int) != calendar_day).sum())
if day_mismatches:
    print(f"Advertencia: {day_mismatches} filas tenían un día inconsistente; se corrigió desde la fecha.")
df["dia_semana"] = calendar_day.astype(int)

calculated_score = df["reacciones"] + 2 * df["comentarios"] + df["clicks"]
score_mismatches = int((df["engagement_score"] != calculated_score).sum())
if score_mismatches:
    print(f"Advertencia: se recalcularon {score_mismatches} scores inconsistentes.")
df["engagement_score"] = calculated_score.astype(float)
df = df.sort_values(["timestamp"], kind="stable").reset_index(drop=True)

if len(df) < 100:
    raise ValueError("Se requieren al menos 100 registros para esta configuración experimental.")

print(f"Registros válidos: {len(df):,}")
print("Periodo:", df["timestamp"].min(), "→", df["timestamp"].max())
display(df.head())

In [ ]:
display(df.describe(include="all").T)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(df["engagement_score"], bins=30, kde=True, ax=axes[0], color="#5b2b76")
axes[0].set_title("Distribución del engagement score")
slot_profile = df.groupby(["dia_semana", "hora_publicacion"])["engagement_score"].agg(["mean", "count"]).reset_index()
pivot = slot_profile.pivot(index="dia_semana", columns="hora_publicacion", values="mean")
sns.heatmap(pivot, cmap="viridis", ax=axes[1])
axes[1].set_title("Promedio observado por día y hora")
axes[1].set_ylabel("Día (0=lunes)")
plt.tight_layout()
plt.show()

display(slot_profile.sort_values("count").head(10))

## 3. Ingeniería de características

Las magnitudes se transforman con `log1p` para reducir el efecto de valores extremos. Día y hora se codifican de forma circular para que domingo/lunes y 23:00/00:00 permanezcan cercanos.

In [ ]:
df["hora_sin"] = np.sin(2 * np.pi * df["hora_publicacion"] / 24)
df["hora_cos"] = np.cos(2 * np.pi * df["hora_publicacion"] / 24)
df["dia_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
df["dia_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
df["gap_hours"] = df["timestamp"].diff().dt.total_seconds().div(3600).clip(lower=0, upper=24 * 30)
df["gap_hours"] = df["gap_hours"].fillna(24.0)

for source, destination in {
    "reacciones": "reacciones_log",
    "comentarios": "comentarios_log",
    "clicks": "clicks_log",
    "usuarios_interactuaron": "usuarios_log",
    "engagement_score": "engagement_log",
    "gap_hours": "gap_log",
}.items():
    df[destination] = np.log1p(df[source].astype(float))

HISTORY_FEATURES = [
    "reacciones_log", "comentarios_log", "clicks_log",
    "usuarios_log", "engagement_log",
    "hora_sin", "hora_cos", "dia_sin", "dia_cos", "gap_log",
]
CANDIDATE_FEATURES = ["hora_sin", "hora_cos", "dia_sin", "dia_cos", "gap_log"]
TARGET_COLUMN = "engagement_log"

history_raw = df[HISTORY_FEATURES].to_numpy(dtype=np.float32)
candidate_raw = df[CANDIDATE_FEATURES].to_numpy(dtype=np.float32)
target_raw = df[[TARGET_COLUMN]].to_numpy(dtype=np.float32)
print("Características históricas:", HISTORY_FEATURES)
print("Características del candidato:", CANDIDATE_FEATURES)

## 4. Secuencias y partición cronológica

Cada muestra usa las 7 publicaciones anteriores para predecir la siguiente. No se mezclan aleatoriamente registros futuros con entrenamiento.

In [ ]:
def build_sequences(history_matrix, candidate_matrix, target_matrix, window):
    histories, candidates, targets, target_indices = [], [], [], []
    for target_index in range(window, len(history_matrix)):
        histories.append(history_matrix[target_index - window:target_index])
        candidates.append(candidate_matrix[target_index])
        targets.append(target_matrix[target_index])
        target_indices.append(target_index)
    return (
        np.asarray(histories, dtype=np.float32),
        np.asarray(candidates, dtype=np.float32),
        np.asarray(targets, dtype=np.float32),
        np.asarray(target_indices, dtype=np.int32),
    )

X_history_raw, X_candidate_raw, y_raw, target_indices = build_sequences(
    history_raw, candidate_raw, target_raw, WINDOW
)
n_samples = len(y_raw)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

train_mask = np.arange(n_samples) < train_end
val_mask = (np.arange(n_samples) >= train_end) & (np.arange(n_samples) < val_end)
test_mask = np.arange(n_samples) >= val_end

# Ajustar cada fila histórica una sola vez y únicamente hasta el último objetivo de entrenamiento.
last_train_target_index = int(target_indices[train_mask][-1])
scaler_history = StandardScaler().fit(history_raw[:last_train_target_index])
scaler_candidate = StandardScaler().fit(X_candidate_raw[train_mask])
scaler_target = StandardScaler().fit(y_raw[train_mask])

def transform_history(values):
    shape = values.shape
    return scaler_history.transform(values.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)

X_history = transform_history(X_history_raw)
X_candidate = scaler_candidate.transform(X_candidate_raw).astype(np.float32)
y = scaler_target.transform(y_raw).astype(np.float32)

Xh_train, Xc_train, y_train = X_history[train_mask], X_candidate[train_mask], y[train_mask]
Xh_val, Xc_val, y_val = X_history[val_mask], X_candidate[val_mask], y[val_mask]
Xh_test, Xc_test, y_test = X_history[test_mask], X_candidate[test_mask], y[test_mask]

def period_for(mask):
    dates = df.iloc[target_indices[mask]]["timestamp"]
    return f"{dates.min()} → {dates.max()}"

print("Entrenamiento:", len(y_train), period_for(train_mask))
print("Validación:   ", len(y_val), period_for(val_mask))
print("Prueba:       ", len(y_test), period_for(test_mask))
print("Forma LSTM:", Xh_train.shape, "| Forma candidato:", Xc_train.shape)

## 5. Modelo LSTM con contexto candidato

La rama LSTM resume el rendimiento histórico. La segunda rama representa la franja que se desea evaluar. Ambas se combinan para estimar el rendimiento de esa publicación candidata.

In [ ]:
tf.keras.backend.clear_session()

history_input = layers.Input(shape=(WINDOW, len(HISTORY_FEATURES)), name="history_sequence")
history_branch = layers.LSTM(48, dropout=0.15, recurrent_dropout=0.0, name="history_lstm")(history_input)
history_branch = layers.Dense(24, activation="relu", name="history_embedding")(history_branch)

candidate_input = layers.Input(shape=(len(CANDIDATE_FEATURES),), name="candidate_slot")
candidate_branch = layers.Dense(16, activation="relu", name="candidate_embedding")(candidate_input)

combined = layers.Concatenate(name="history_candidate_merge")([history_branch, candidate_branch])
combined = layers.Dense(32, activation="relu")(combined)
combined = layers.Dropout(0.20)(combined)
combined = layers.Dense(16, activation="relu")(combined)
output = layers.Dense(1, name="predicted_engagement")(combined)

model = Model(inputs=[history_input, candidate_input], outputs=output, name="publication_time_lstm_v2")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model.summary()

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = ARTIFACT_DIR / "modelo_lstm_horarios_v2.keras"

callbacks = [
    EarlyStopping(monitor="val_loss", patience=18, min_delta=1e-4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5, verbose=1),
    ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True, verbose=1),
]

history = model.fit(
    {"history_sequence": Xh_train, "candidate_slot": Xc_train},
    y_train,
    validation_data=({"history_sequence": Xh_val, "candidate_slot": Xc_val}, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1,
)
print("Mejor modelo guardado en:", best_model_path)

In [ ]:
training_history = pd.DataFrame(history.history)
best_epoch = int(training_history["val_loss"].idxmin() + 1)
display(training_history.tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(training_history["loss"], label="Entrenamiento")
axes[0].plot(training_history["val_loss"], label="Validación")
axes[0].axvline(best_epoch - 1, color="#ef6c22", linestyle="--", label=f"Mejor época: {best_epoch}")
axes[0].set_title("MSE durante el entrenamiento")
axes[0].legend()
axes[1].plot(training_history["mae"], label="Entrenamiento")
axes[1].plot(training_history["val_mae"], label="Validación")
axes[1].set_title("MAE escalado durante el entrenamiento")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Evaluación contra modelos base

Una LSTM no se considera útil solamente por tener capas recurrentes. Debe compararse con alternativas simples calculadas sin utilizar el periodo de prueba.

In [ ]:
model = tf.keras.models.load_model(best_model_path)

def inverse_target(scaled_values):
    log_values = scaler_target.inverse_transform(np.asarray(scaled_values).reshape(-1, 1)).ravel()
    return np.maximum(np.expm1(log_values), 0.0)

def regression_metrics(actual, predicted):
    correlation = spearmanr(actual, predicted).statistic
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)),
        "Spearman": float(correlation) if np.isfinite(correlation) else 0.0,
    }

actual_test = inverse_target(y_test)
pred_lstm_scaled = model.predict(
    {"history_sequence": Xh_test, "candidate_slot": Xc_test}, verbose=0
)
pred_lstm = inverse_target(pred_lstm_scaled)

train_rows = df.iloc[target_indices[train_mask]].copy()
test_rows = df.iloc[target_indices[test_mask]].copy()
global_train_mean = float(train_rows["engagement_score"].mean())
slot_train_means = train_rows.groupby(["dia_semana", "hora_publicacion"])["engagement_score"].mean().to_dict()
pred_global = np.full_like(actual_test, global_train_mean, dtype=float)
pred_slot = np.array([
    slot_train_means.get((int(row.dia_semana), int(row.hora_publicacion)), global_train_mean)
    for row in test_rows.itertuples()
])

metric_rows = []
for name, prediction in {
    "Media global": pred_global,
    "Promedio día/hora": pred_slot,
    "LSTM v2": pred_lstm,
}.items():
    metric_rows.append({"modelo": name, **regression_metrics(actual_test, prediction)})
metrics_df = pd.DataFrame(metric_rows).sort_values("MAE").reset_index(drop=True)
display(metrics_df.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}", "Spearman": "{:.3f}"}))

lstm_mae = float(metrics_df.loc[metrics_df["modelo"] == "LSTM v2", "MAE"].iloc[0])
best_baseline_mae = float(metrics_df.loc[metrics_df["modelo"] != "LSTM v2", "MAE"].min())
if lstm_mae < best_baseline_mae:
    print(f"Resultado favorable: la LSTM reduce el MAE en {(best_baseline_mae - lstm_mae) / best_baseline_mae * 100:.2f}% frente al mejor modelo base.")
else:
    print("Advertencia metodológica: la LSTM todavía no supera al mejor modelo base. No debe afirmarse que es superior sin mejorar datos o configuración.")

In [ ]:
comparison = pd.DataFrame({
    "timestamp": test_rows["timestamp"].to_numpy(),
    "real": actual_test,
    "lstm": pred_lstm,
    "promedio_dia_hora": pred_slot,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(comparison["timestamp"], comparison["real"], label="Real", alpha=0.75)
axes[0].plot(comparison["timestamp"], comparison["lstm"], label="LSTM", alpha=0.85)
axes[0].set_title("Prueba cronológica: real vs. LSTM")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=30)
axes[1].scatter(comparison["real"], comparison["lstm"], alpha=0.65, color="#117e8c")
limit = max(comparison["real"].max(), comparison["lstm"].max())
axes[1].plot([0, limit], [0, limit], linestyle="--", color="#ef6c22")
axes[1].set_xlabel("Engagement real")
axes[1].set_ylabel("Engagement predicho")
axes[1].set_title("Calibración visual")
plt.tight_layout()
plt.show()
display(comparison.head(20))

## 7. Evaluar franjas futuras y generar el ranking

La siguiente función conserva la misma secuencia histórica y cambia el día/hora candidato. Solamente evalúa horas observadas durante el entrenamiento, evitando extrapolar a franjas desconocidas. En producción, Laravel enviará la secuencia de cada cuenta y las fechas futuras disponibles.

In [ ]:
DAY_NAMES = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
observed_hours = sorted(train_rows["hora_publicacion"].astype(int).unique().tolist())

def candidate_features(candidate_timestamp, last_timestamp):
    hour = candidate_timestamp.hour
    day = candidate_timestamp.dayofweek
    gap_hours = max((candidate_timestamp - last_timestamp).total_seconds() / 3600, 0)
    gap_hours = min(gap_hours, 24 * 30)
    return [
        np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * day / 7), np.cos(2 * np.pi * day / 7),
        np.log1p(gap_hours),
    ]

def rank_future_slots(history_rows, last_timestamp, days_ahead=7, hours=None):
    if len(history_rows) != WINDOW:
        raise ValueError(f"Se esperaban exactamente {WINDOW} publicaciones históricas.")
    hours = observed_hours if hours is None else list(hours)
    candidate_records = []
    first_day = last_timestamp.normalize()
    for offset in range(days_ahead + 1):
        day = first_day + pd.Timedelta(days=offset)
        for hour in hours:
            timestamp = day + pd.Timedelta(hours=int(hour))
            if timestamp > last_timestamp:
                candidate_records.append({
                    "timestamp": timestamp,
                    "dia_semana": int(timestamp.dayofweek),
                    "hora": int(hour),
                    "features": candidate_features(timestamp, last_timestamp),
                })
    raw_candidates = np.asarray([item["features"] for item in candidate_records], dtype=np.float32)
    scaled_candidates = scaler_candidate.transform(raw_candidates).astype(np.float32)
    raw_history = history_rows[HISTORY_FEATURES].to_numpy(dtype=np.float32)[None, :, :]
    scaled_history = transform_history(raw_history)
    repeated_history = np.repeat(scaled_history, len(candidate_records), axis=0)
    predictions_scaled = model.predict(
        {"history_sequence": repeated_history, "candidate_slot": scaled_candidates}, verbose=0
    )
    predictions = inverse_target(predictions_scaled)
    ranking = pd.DataFrame([{
        "timestamp": item["timestamp"],
        "dia_semana": item["dia_semana"],
        "dia": DAY_NAMES[item["dia_semana"]],
        "hora": f"{item['hora']:02d}:00",
        "engagement_predicho": float(prediction),
    } for item, prediction in zip(candidate_records, predictions)])
    return ranking.sort_values("engagement_predicho", ascending=False).reset_index(drop=True)

last_timestamp = pd.Timestamp(df["timestamp"].iloc[-1])
latest_history = df.iloc[-WINDOW:].copy()
future_ranking = rank_future_slots(latest_history, last_timestamp, days_ahead=7)
display(future_ranking.head(10).style.format({"engagement_predicho": "{:.2f}"}))

## 8. Exportar paquete reproducible

El ZIP contiene todo lo necesario para reproducir el preprocesamiento y servir predicciones. No contiene tokens, túneles ni credenciales.

In [ ]:
training_history.to_csv(ARTIFACT_DIR / "historial_entrenamiento.csv", index=False)
metrics_df.to_csv(ARTIFACT_DIR / "metricas_prueba.csv", index=False)
comparison.to_csv(ARTIFACT_DIR / "predicciones_prueba.csv", index=False)
future_ranking.to_csv(ARTIFACT_DIR / "ranking_demostracion.csv", index=False)
joblib.dump(scaler_history, ARTIFACT_DIR / "scaler_history.joblib")
joblib.dump(scaler_candidate, ARTIFACT_DIR / "scaler_candidate.joblib")
joblib.dump(scaler_target, ARTIFACT_DIR / "scaler_target.joblib")
model.save(ARTIFACT_DIR / "modelo_lstm_horarios_v2.keras")

lstm_metrics = metrics_df.loc[metrics_df["modelo"] == "LSTM v2"].iloc[0].to_dict()
metadata = {
    "model_version": MODEL_VERSION,
    "model_type": "LSTM encoder + candidate context",
    "platform": "facebook",
    "window": WINDOW,
    "history_features": HISTORY_FEATURES,
    "candidate_features": CANDIDATE_FEATURES,
    "target": "engagement_score",
    "target_transformation": "log1p then StandardScaler",
    "engagement_formula": "reacciones + 2 * comentarios + clicks",
    "day_convention": "0=lunes, 6=domingo",
    "timezone_expected": "America/La_Paz",
    "observed_hours": observed_hours,
    "dataset_rows": int(len(df)),
    "dataset_start": df["timestamp"].min().isoformat(),
    "dataset_end": df["timestamp"].max().isoformat(),
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "strategy": "chronological"},
    "best_epoch": best_epoch,
    "test_metrics": {key: float(value) for key, value in lstm_metrics.items() if key != "modelo"},
    "limitations": [
        "El dataset no identifica cuentas individuales.",
        "El dataset actual representa Facebook, no Instagram.",
        "No existe alcance por publicación; se predice engagement absoluto.",
        "La confianza personalizada depende del histórico real disponible por cuenta.",
    ],
}
with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

example_input = {
    "model_version": MODEL_VERSION,
    "timezone": "America/La_Paz",
    "history_raw": latest_history[HISTORY_FEATURES].to_dict(orient="records"),
    "candidate_raw": {
        name: float(value)
        for name, value in zip(CANDIDATE_FEATURES, candidate_features(future_ranking.iloc[0]["timestamp"], last_timestamp))
    },
}
with open(ARTIFACT_DIR / "ejemplo_entrada_inferencia.json", "w", encoding="utf-8") as file:
    json.dump(example_input, file, ensure_ascii=False, indent=2)

print("Artefactos generados:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name, f"({path.stat().st_size:,} bytes)")

In [ ]:
archive_base = Path("/content/lstm_horarios_facebook_v2")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=ARTIFACT_DIR)
print("Paquete final:", archive_path)
print("Tamaño:", f"{Path(archive_path).stat().st_size / 1024 / 1024:.2f} MB")
files.download(archive_path)

## Criterio para continuar con la integración

Antes de conectar el modelo con Laravel, conserva estos resultados:

1. Tabla completa de métricas.
2. Mensaje que indica si la LSTM superó al mejor modelo base.
3. Gráficas de entrenamiento y prueba.
4. Archivo `lstm_horarios_facebook_v2.zip`.

Si la LSTM no supera el modelo base, el cuaderno sigue siendo válido como experimento, pero se deberá ajustar la arquitectura o enriquecer el dataset antes de usarla como recomendador principal. No se deben alterar o esconder métricas desfavorables.